In [66]:
import json
import requests
from pathlib import Path

# Folder where all your lakes sources live
SRC = Path("../sources").resolve()

raw_path = SRC / "osm_berlin_lakes_raw.json"
geojson_path = SRC / "osm_berlin_lakes.geojson"

# Overpass API endpoint (the "link")
overpass_url = "https://overpass-api.de/api/interpreter"

# ✅ Corrected Overpass query with geometry output
query = """
[out:json][timeout:180];
area[name="Berlin"]["boundary"="administrative"]->.searchArea;

(
  way["natural"="water"]["water"~"lake|pond"](area.searchArea);
  relation["natural"="water"]["water"~"lake|pond"](area.searchArea);
);

out geom;
"""

print("Fetching OSM lakes from Overpass…")
resp = requests.get(overpass_url, params={"data": query})
resp.raise_for_status()

data = resp.json()

# 1) Save raw Overpass JSON
with raw_path.open("w", encoding="utf-8") as f:
    json.dump(data, f)
print("✅ OSM raw data saved to:", raw_path)

# 2) Build a simple GeoJSON FeatureCollection from the raw JSON
elements = data.get("elements", [])
features = []

for el in elements:
    geom = None

    # We focus on ways with geometry (lake outlines)
    if el.get("type") == "way" and "geometry" in el:
        coords = [(p["lon"], p["lat"]) for p in el["geometry"]]

        # Close polygon if needed
        if len(coords) >= 4 and coords[0] != coords[-1]:
            coords.append(coords[0])

        geom = {
            "type": "Polygon",
            "coordinates": [coords],
        }

    if geom is None:
        continue

    tags = el.get("tags", {})
    props = {
        "name": tags.get("name"),
        "natural": tags.get("natural"),
        "water": tags.get("water"),
        "wikidata": tags.get("wikidata"),
    }

    features.append({
        "type": "Feature",
        "geometry": geom,
        "properties": props,
    })

geojson = {
    "type": "FeatureCollection",
    "name": "osm_berlin_lakes",
    "features": features,
}

with geojson_path.open("w", encoding="utf-8") as f:
    json.dump(geojson, f)

print(f"✅ GeoJSON saved with {len(features)} features to:", geojson_path)


Fetching OSM lakes from Overpass…
✅ OSM raw data saved to: /Users/robertsesazi/Documents/layered-populate-data-pool-da/lakes/sources/osm_berlin_lakes_raw.json
✅ GeoJSON saved with 633 features to: /Users/robertsesazi/Documents/layered-populate-data-pool-da/lakes/sources/osm_berlin_lakes.geojson


In [22]:
from pathlib import Path
import pandas as pd
import geopandas as gpd

SRC = Path("../sources").resolve()
csv_path = SRC / "berlin_lakes_summary.csv"
geo_path = SRC / "berlin_lakes_clean.geojson"

print("Reading CSV from:", csv_path)

# Let pandas auto-detect the separator (or set sep=";" if you know it's ; )
df = pd.read_csv(csv_path, sep=None, engine="python")

print("Raw CSV shape:", df.shape)
print("Raw columns:", df.columns.tolist())

# Clean column names (remove spaces etc.)
df.columns = [c.strip() for c in df.columns]
print("Cleaned columns:", df.columns.tolist())

# ---- make sure coordinate columns exist ----
coord_cols = ["centroid_lon", "centroid_lat"]
missing = [c for c in coord_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing coordinate columns: {missing}")

# ---- drop rows without coordinates ----
before = df.shape[0]
df = df.dropna(subset=coord_cols)
print(f"Removed {before - df.shape[0]} rows without coordinates.")
print("Remaining rows:", df.shape[0])

# ---- build GeoDataFrame from centroids ----
gdf = gpd.GeoDataFrame(
    df.copy(),
    geometry=gpd.points_from_xy(df["centroid_lon"], df["centroid_lat"]),
    crs="EPSG:4326",
)

print("GeoDataFrame shape:", gdf.shape)

# ---- save GeoJSON (does NOT touch any other files) ----
gdf.to_file(geo_path, driver="GeoJSON")
print("✅ Saved berlin_lakes_clean.geojson to:", geo_path)


Reading CSV from: /Users/robertsesazi/Documents/layered-populate-data-pool-da/lakes/sources/berlin_lakes_summary.csv
Raw CSV shape: (276, 3)
Raw columns: ['centroid_lon', 'centroid_lat', 'source']
Cleaned columns: ['centroid_lon', 'centroid_lat', 'source']
Removed 0 rows without coordinates.
Remaining rows: 276
GeoDataFrame shape: (276, 4)
✅ Saved berlin_lakes_clean.geojson to: /Users/robertsesazi/Documents/layered-populate-data-pool-da/lakes/sources/berlin_lakes_clean.geojson


In [23]:
%pip install -q --upgrade pandas geopandas pyogrio shapely fiona


Note: you may need to restart the kernel to use updated packages.


In [24]:
import os, glob, pathlib
SRC = pathlib.Path("../sources").resolve()
print("Looking in:", SRC)
files = sorted(glob.glob(str(SRC / "*")))
for f in files:
    print("-", os.path.basename(f))


Looking in: /Users/robertsesazi/Documents/layered-populate-data-pool-da/lakes/sources
- README.md
- berlin_lakes_clean.geojson
- berlin_lakes_summary.csv
- demeritzsee.csv
- demeritzsee_clean.csv
- lakes_berlin_unified.geojson
- osm_berlin_lakes.geojson
- osm_berlin_lakes_clean.geojson
- osm_berlin_lakes_raw.json
- scripts


In [25]:
from pathlib import Path
import geopandas as gpd

SRC = Path("../sources").resolve()
# use the cleaned GeoJSON that we just created
geo_path = SRC / "berlin_lakes_clean.geojson"

print("Looking for:", geo_path)
print("Exists:", geo_path.exists())

if not geo_path.exists():
    raise FileNotFoundError(f"Cannot find file at: {geo_path}")

gdf = gpd.read_file(geo_path)
print("✅ GeoDataFrame loaded")
print("Rows:", len(gdf))
print("Columns:", list(gdf.columns))
print("CRS:", gdf.crs)
gdf.head(3)


Looking for: /Users/robertsesazi/Documents/layered-populate-data-pool-da/lakes/sources/berlin_lakes_clean.geojson
Exists: True
✅ GeoDataFrame loaded
Rows: 276
Columns: ['centroid_lon', 'centroid_lat', 'source', 'geometry']
CRS: EPSG:4326


,centroid_lon,centroid_lat,source,geometry
0,13.278930,52.543856,OSM lakes,POINT (13.27893 52.54386)
1,13.219207,52.550213,OSM lakes,POINT (13.21921 52.55021)
2,13.280684,52.486131,OSM lakes,POINT (13.28068 52.48613)


In [26]:
# Reloading the CSV with semicolon separator
df = pd.read_csv(csv_path, sep=";", skiprows=5)

print("✅ CSV reloaded:", df.shape)
print("Columns:", df.columns.tolist())
df.head(10)


✅ CSV reloaded: (271, 1)
Columns: ['13.625819492700556,52.400747119279224,OSM lakes']


,"13.625819492700556,52.400747119279224,OSM lakes"
0,"13.342409843277231,52.511382596608854,OSM lakes"
1,"13.360881486397284,52.56442907729419,OSM lakes"
2,"13.540301000022128,52.529139119876554,OSM lakes"
3,"13.112259399844673,52.46537373970797,OSM lakes"
4,"13.127598056517185,52.43629415166612,OSM lakes"
5,"13.489121497880472,52.548601247220105,OSM lakes"
6,"13.641019635973793,52.35693290759696,OSM lakes"
7,"13.308928578658268,52.597672433485215,OSM lakes"
8,"13.212941487280196,52.51928350862598,OSM lakes"
9,"13.417134638368406,52.43427708454176,OSM lakes"


In [28]:
# --- Safely drop meta/header rows and empty columns ---

# Some rows in "dt" contain text like "water physics" or "YYYY-MM-DD".
# We only try this if a suitable time column exists.
time_col = None
if "dt" in df.columns:
    time_col = "dt"
elif "datetime" in df.columns:
    time_col = "datetime"

if time_col is not None:
    # Identify meta/header rows
    mask_meta = df[time_col].astype(str).str.contains(
        "water physics|YYYY", na=False
    )
    removed = mask_meta.sum()
    rows_before = len(df)

    # Keep only non-meta rows
    df = df[~mask_meta].copy()

    print(f"✅ Removed {removed} meta/header rows "
          f"(from {rows_before} to {len(df)}).")
else:
    print("ℹ️ No 'dt' or 'datetime' column found – skipping meta row removal.")

# Drop completely empty columns (this is safe)
df = df.dropna(axis=1, how="all")

print("✅ Cleaned headers, remaining rows:", df.shape)
print("Columns now:", df.columns.tolist())
df.head(10)


ℹ️ No 'dt' or 'datetime' column found – skipping meta row removal.
✅ Cleaned headers, remaining rows: (271, 1)
Columns now: ['13.625819492700556,52.400747119279224,OSM lakes']


,"13.625819492700556,52.400747119279224,OSM lakes"
0,"13.342409843277231,52.511382596608854,OSM lakes"
1,"13.360881486397284,52.56442907729419,OSM lakes"
2,"13.540301000022128,52.529139119876554,OSM lakes"
3,"13.112259399844673,52.46537373970797,OSM lakes"
4,"13.127598056517185,52.43629415166612,OSM lakes"
5,"13.489121497880472,52.548601247220105,OSM lakes"
6,"13.641019635973793,52.35693290759696,OSM lakes"
7,"13.308928578658268,52.597672433485215,OSM lakes"
8,"13.212941487280196,52.51928350862598,OSM lakes"
9,"13.417134638368406,52.43427708454176,OSM lakes"


In [29]:
rename_map = {
    'dt': 'datetime',
    'w_temp': 'temp_c',
    'cond': 'conductivity_µS',
    'depth': 'depth_m',
    'pH': 'pH',
    'o2_sat': 'oxygen_saturation_pct',
    'o2_con': 'oxygen_concentration_mgL',
    'secchi': 'secchi_depth_m'
}
df = df.rename(columns={c: rename_map.get(c, c) for c in df.columns})
print("✅ Columns renamed:")
df.head(3)


✅ Columns renamed:


,"13.625819492700556,52.400747119279224,OSM lakes"
0,"13.342409843277231,52.511382596608854,OSM lakes"
1,"13.360881486397284,52.56442907729419,OSM lakes"
2,"13.540301000022128,52.529139119876554,OSM lakes"


In [31]:
from pathlib import Path
import pandas as pd

SRC = Path("../sources").resolve()
raw_path = SRC / "demeritzsee.csv"
out_csv = SRC / "demeritzsee_clean.csv"

# 1) Load raw file – skip the metadata lines at the top
df_dem = pd.read_csv(raw_path, sep=";", skiprows=5)

print("Raw Dämmeritzsee loaded:", df_dem.shape)
print("Columns:", df_dem.columns.tolist())

# 2) Keep only real measurement rows (dt starts with a date like 1992-04-30 …)
mask_measurements = df_dem["dt"].astype(str).str.match(r"\d{4}-\d{2}-\d{2}", na=False)
df_dem = df_dem[mask_measurements].copy()
print("After keeping only measurements:", df_dem.shape)

# 3) Rename columns to nice names
rename_map = {
    "dt": "datetime",
    "w_temp": "temp_c",
    "cond": "conductivity_µS",
    "depth": "depth_m",
    "pH": "pH",
    "o2_sat": "oxygen_saturation_pct",
    "o2_con": "oxygen_concentration_mgL",
    "secchi": "secchi_depth_m",
}
df_dem = df_dem.rename(columns={c: rename_map.get(c, c) for c in df_dem.columns})
print("Columns renamed:", df_dem.columns.tolist())

# 4) Convert datetime and numeric columns
df_dem["datetime"] = pd.to_datetime(df_dem["datetime"], errors="coerce")
df_dem = df_dem[df_dem["datetime"].notna()].copy()

numeric_cols = [
    "temp_c",
    "conductivity_µS",
    "depth_m",
    "pH",
    "oxygen_saturation_pct",
    "oxygen_concentration_mgL",
    "secchi_depth_m",
]
for col in numeric_cols:
    if col in df_dem.columns:
        df_dem[col] = pd.to_numeric(df_dem[col], errors="coerce")

print("Converted types:")
print(df_dem.dtypes)

# 5) Add lake_name and source columns
df_dem["lake_name"] = "Dämeritzsee"
df_dem["source"] = "In-situ measurement"

print("Clean DataFrame ready:", df_dem.shape)
df_dem.head(5)

# 6) Save clean CSV
df_dem.to_csv(out_csv, index=False, sep=";")
print("✅ Saved clean Dämmeritzsee CSV to:", out_csv)


Raw Dämmeritzsee loaded: (198, 9)
Columns: ['dt', 'w_temp', 'cond', 'depth', 'pH', 'o2_sat', 'o2_con', 'secchi', 'Unnamed: 8']
After keeping only measurements: (194, 9)
Columns renamed: ['datetime', 'temp_c', 'conductivity_µS', 'depth_m', 'pH', 'oxygen_saturation_pct', 'oxygen_concentration_mgL', 'secchi_depth_m', 'Unnamed: 8']
Converted types:
datetime                    datetime64[ns]
temp_c                             float64
conductivity_µS                    float64
depth_m                            float64
pH                                 float64
oxygen_saturation_pct              float64
oxygen_concentration_mgL           float64
secchi_depth_m                     float64
Unnamed: 8                          object
dtype: object
Clean DataFrame ready: (194, 11)
✅ Saved clean Dämmeritzsee CSV to: /Users/robertsesazi/Documents/layered-populate-data-pool-da/lakes/sources/demeritzsee_clean.csv


In [33]:
df['lake_name'] = "Dämeritzsee"
df['source'] = "In-situ measurement"

print("✅ Clean DataFrame ready:", df.shape)
df.head(5)


✅ Clean DataFrame ready: (271, 3)


,"13.625819492700556,52.400747119279224,OSM lakes",lake_name,source
0,"13.342409843277231,52.511382596608854,OSM lakes",Dämeritzsee,In-situ measurement
1,"13.360881486397284,52.56442907729419,OSM lakes",Dämeritzsee,In-situ measurement
2,"13.540301000022128,52.529139119876554,OSM lakes",Dämeritzsee,In-situ measurement
3,"13.112259399844673,52.46537373970797,OSM lakes",Dämeritzsee,In-situ measurement
4,"13.127598056517185,52.43629415166612,OSM lakes",Dämeritzsee,In-situ measurement


In [35]:
clean_csv_path = Path("../sources/demeritzsee_clean.csv")
df.to_csv(clean_csv_path, index=False)
print("💾 Saved cleaned dataset to:", clean_csv_path)


💾 Saved cleaned dataset to: ../sources/demeritzsee_clean.csv


In [34]:
import geopandas as gpd
from pathlib import Path

geo_path = Path("../sources/osm_berlin_lakes.geojson")

print("Looking for:", geo_path)
print("Exists:", geo_path.exists())

if not geo_path.exists():
    raise FileNotFoundError(f"Cannot find file at: {geo_path}")

gdf = gpd.read_file(geo_path)

print("GeoDataFrame loaded:", gdf.shape)
print("Columns:", list(gdf.columns))
print("CRS:", gdf.crs)
gdf.head(3)


Looking for: ../sources/osm_berlin_lakes.geojson
Exists: True
GeoDataFrame loaded: (633, 5)
Columns: ['name', 'natural', 'water', 'wikidata', 'geometry']
CRS: EPSG:4326


,name,natural,water,wikidata,geometry
0,Schlachtensee,water,lake,Q896095,"POLYGON ((13.2281 52.44468, 13.22814 52.44463,..."
1,Koenigssee,water,lake,Q1778289,"POLYGON ((13.27181 52.48856, 13.27203 52.48864..."
2,Fennsee,water,lake,Q1404799,"POLYGON ((13.31548 52.48379, 13.31556 52.48379..."


In [230]:
# Keeping only key columns
cols_to_keep = ['name', 'natural', 'water', 'wikidata', 'geometry']

existing = [c for c in cols_to_keep if c in gdf.columns]
missing = [c for c in cols_to_keep if c not in gdf.columns]

print("Keeping:", existing)
print("Missing (ignored):", missing)

gdf_clean = gdf[existing].copy()

print("Cleaned OSM shape:", gdf_clean.shape)
gdf_clean.head(3)


Keeping: ['name', 'natural', 'water', 'wikidata', 'geometry']
Missing (ignored): []
Cleaned OSM shape: (1908, 5)


,name,natural,water,wikidata,geometry
0,Jungfernheideteich,water,pond,Q63887019,"POLYGON ((13.27588 52.54329, 13.27594 52.54328..."
1,Wiesenteich,water,None,None,"POLYGON ((13.09516 52.41275, 13.09533 52.41267..."
2,Spandauer See,water,lake,Q63284050,"POLYGON ((13.20902 52.54121, 13.20901 52.54125..."


In [37]:
# Keeping only key columns
cols_to_keep = ['name', 'natural', 'water', 'wikidata', 'geometry']
gdf = gdf[[c for c in cols_to_keep if c in gdf.columns]]

# Removeing unnamed water bodies (no name)
gdf = gdf[gdf['name'].notna()].reset_index(drop=True)

# Droping duplicates by name
gdf = gdf.drop_duplicates(subset='name')

print("✅ Cleaned GeoDataFrame:", gdf.shape)
gdf.head(10)


✅ Cleaned GeoDataFrame: (241, 5)


,name,natural,water,wikidata,geometry
0,Schlachtensee,water,lake,Q896095,"POLYGON ((13.2281 52.44468, 13.22814 52.44463,..."
1,Koenigssee,water,lake,Q1778289,"POLYGON ((13.27181 52.48856, 13.27203 52.48864..."
2,Fennsee,water,lake,Q1404799,"POLYGON ((13.31548 52.48379, 13.31556 52.48379..."
3,Dreipfuhl,water,pond,None,"POLYGON ((13.27313 52.44797, 13.27316 52.44798..."
4,Lietzensee,water,lake,Q258569,"POLYGON ((13.289 52.50765, 13.28874 52.50756, ..."
5,Dianasee,water,lake,Q1208924,"POLYGON ((13.26827 52.48627, 13.26769 52.48601..."
6,Hundekehlesee,water,lake,Q1263106,"POLYGON ((13.25745 52.48197, 13.25737 52.48216..."
7,Grunewaldsee,water,lake,Q895938,"POLYGON ((13.25713 52.4678, 13.25717 52.46784,..."
8,Kleiner Wannsee,water,lake,Q453692,"POLYGON ((13.15402 52.41227, 13.15406 52.41239..."
9,Stölpchensee,water,lake,Q2360565,"POLYGON ((13.1429 52.41028, 13.1428 52.41023, ..."


In [41]:
# Save cleaned OSM lakes GeoJSON
from pathlib import Path  # ok if this is imported earlier as well

clean_osm_path = Path("../sources/osm_berlin_lakes_clean.geojson")

print("Saving cleaned OSM lakes GeoJSON to:", clean_osm_path)
print("GeoDataFrame shape:", gdf.shape)
print("Columns:", list(gdf.columns))

# gdf is the already-cleaned GeoDataFrame from the previous cell
gdf.to_file(clean_osm_path, driver="GeoJSON")

print("✅ Saved cleaned OSM lakes to:", clean_osm_path)


Saving cleaned OSM lakes GeoJSON to: ../sources/osm_berlin_lakes_clean.geojson
GeoDataFrame shape: (241, 5)
Columns: ['name', 'natural', 'water', 'wikidata', 'geometry']
✅ Saved cleaned OSM lakes to: ../sources/osm_berlin_lakes_clean.geojson


In [42]:
# Searchinf for Dämeritzsee in the OSM data
mask = gdf['name'].str.contains("Dämeritz", case=False, na=False)
gdf_lake = gdf[mask]

print("✅ Found matching lake(s):", len(gdf_lake))
gdf_lake


✅ Found matching lake(s): 0


,name,natural,water,wikidata,geometry


In [44]:
out_path = Path("../sources/daemeritzsee_polygon.geojson")
gdf_lake.to_file(out_path, driver="GeoJSON")
print("💾 Saved Dämeritzsee polygon to:", out_path)


💾 Saved Dämeritzsee polygon to: ../sources/daemeritzsee_polygon.geojson


In [45]:
# --- Clean + enrich OSM lakes (Berlin) ---

import unicodedata

# At this point `gdf` is already loaded from the previous cell.
# We'll work on a copy so we don't overwrite the original accidentally.
gdf = gdf.copy()

# keeping only essential columns (only those that actually exist)
cols_to_keep = [c for c in ["name", "natural", "water", "wikidata", "geometry"] if c in gdf.columns]
gdf = gdf[cols_to_keep].copy()

# keeping water bodies that are lakes/ponds (only if these columns exist)
if "natural" in gdf.columns:
    gdf = gdf[gdf["natural"].fillna("").str.lower().eq("water")]
if "water" in gdf.columns:
    gdf = gdf[gdf["water"].fillna("").str.lower().isin(["lake", "pond"])]

# unamed and duplicates – drop rows without a name, then deduplicate by name
if "name" in gdf.columns:
    gdf = gdf[gdf["name"].notna()].drop_duplicates(subset="name").reset_index(drop=True)

    # adding normalized name for robust matching (strip accents, lowercase)
    def normalize(s):
        s = unicodedata.normalize("NFKD", str(s))
        s = "".join(ch for ch in s if not unicodedata.combining(ch))
        return s.lower().strip()

    gdf["name_norm"] = gdf["name"].apply(normalize)

# project to metric CRS around Berlin for correct areas (EPSG:25833 is UTM33N)
gdf_metric = gdf.to_crs(25833)

# computing area (sq km) and centroids (lat/lon)
gdf_metric["area_sqkm"] = gdf_metric.geometry.area / 1e6
centroids_lonlat = gdf_metric.to_crs(4326).centroid
gdf_metric["centroid_lon"] = centroids_lonlat.x
gdf_metric["centroid_lat"] = centroids_lonlat.y

# bringing back to WGS84 for saving
gdf_clean = gdf_metric.to_crs(4326)

print("✅ Cleaned OSM gdf:", gdf_clean.shape)
gdf_clean.head()


✅ Cleaned OSM gdf: (241, 9)


/var/folders/kl/8b94v_052td7vx7hwzsc0fpw0000gn/T/ipykernel_12615/3050197729.py:36: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  centroids_lonlat = gdf_metric.to_crs(4326).centroid


,name,natural,water,wikidata,geometry,name_norm,area_sqkm,centroid_lon,centroid_lat
0,Schlachtensee,water,lake,Q896095,"POLYGON ((13.2281 52.44468, 13.22814 52.44463,...",schlachtensee,0.403524,13.213323,52.442076
1,Koenigssee,water,lake,Q1778289,"POLYGON ((13.27181 52.48856, 13.27203 52.48864...",koenigssee,0.022272,13.272907,52.488474
2,Fennsee,water,lake,Q1404799,"POLYGON ((13.31548 52.48379, 13.31556 52.48379...",fennsee,0.022057,13.314797,52.483685
3,Dreipfuhl,water,pond,None,"POLYGON ((13.27313 52.44797, 13.27316 52.44798...",dreipfuhl,0.006412,13.272662,52.447221
4,Lietzensee,water,lake,Q258569,"POLYGON ((13.289 52.50765, 13.28874 52.50756, ...",lietzensee,0.063648,13.289096,52.506695


In [48]:
out_geo = Path("../sources/berlin_lakes_clean.geojson")
out_csv = Path("../sources/berlin_lakes_summary.csv")

# GeoJSON
gdf_clean[["name","name_norm","natural","water","wikidata","area_sqkm",
           "centroid_lon","centroid_lat","geometry"]].to_file(out_geo, driver="GeoJSON")

# Saving my CSV without geometry for easy viewing
gdf_clean.drop(columns="geometry").to_csv(out_csv, index=False)

print("💾 Saved:", out_geo)
print("💾 Saved:", out_csv)


💾 Saved: ../sources/berlin_lakes_clean.geojson
💾 Saved: ../sources/berlin_lakes_summary.csv


In [49]:
# --- Clean + enrich OSM lakes (Berlin) ---

import unicodedata

# `gdf` is already loaded from the OSM loader cell above.
# Work on a copy:
gdf = gdf.copy()

# keep only essential columns that actually exist
cols_to_keep = [c for c in ["name", "natural", "water", "wikidata", "geometry"] if c in gdf.columns]
gdf = gdf[cols_to_keep].copy()

# keep only water bodies that look like lakes/ponds
if "natural" in gdf.columns:
    gdf = gdf[gdf["natural"].fillna("").str.lower().eq("water")]

if "water" in gdf.columns:
    gdf = gdf[gdf["water"].fillna("").str.lower().isin(["lake", "pond"])]

# drop unnamed and duplicates
if "name" in gdf.columns:
    gdf = gdf[gdf["name"].notna()].drop_duplicates(subset="name").reset_index(drop=True)

    # normalize name
    def normalize(s):
        s = unicodedata.normalize("NFKD", str(s))
        s = "".join(ch for ch in s if not unicodedata.combining(ch))
        return s.lower().strip()

    gdf["name_norm"] = gdf["name"].apply(normalize)

# project to metric CRS for area + centroid computations
gdf_metric = gdf.to_crs(25833)

gdf_metric["area_sqkm"] = gdf_metric.geometry.area / 1e6

centroids = gdf_metric.to_crs(4326).centroid
gdf_metric["centroid_lon"] = centroids.x
gdf_metric["centroid_lat"] = centroids.y

# convert back to WGS84 for saving
gdf_clean = gdf_metric.to_crs(4326)

print("✅ Cleaned OSM gdf:", gdf_clean.shape)
gdf_clean.head()


✅ Cleaned OSM gdf: (241, 9)


/var/folders/kl/8b94v_052td7vx7hwzsc0fpw0000gn/T/ipykernel_12615/1454723452.py:37: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  centroids = gdf_metric.to_crs(4326).centroid


,name,natural,water,wikidata,geometry,name_norm,area_sqkm,centroid_lon,centroid_lat
0,Schlachtensee,water,lake,Q896095,"POLYGON ((13.2281 52.44468, 13.22814 52.44463,...",schlachtensee,0.403524,13.213323,52.442076
1,Koenigssee,water,lake,Q1778289,"POLYGON ((13.27181 52.48856, 13.27203 52.48864...",koenigssee,0.022272,13.272907,52.488474
2,Fennsee,water,lake,Q1404799,"POLYGON ((13.31548 52.48379, 13.31556 52.48379...",fennsee,0.022057,13.314797,52.483685
3,Dreipfuhl,water,pond,None,"POLYGON ((13.27313 52.44797, 13.27316 52.44798...",dreipfuhl,0.006412,13.272662,52.447221
4,Lietzensee,water,lake,Q258569,"POLYGON ((13.289 52.50765, 13.28874 52.50756, ...",lietzensee,0.063648,13.289096,52.506695


In [50]:
candidates = ["dämeritzsee", "daemeritzsee", "dameritzsee"]

mask = gdf_clean["name_norm"].apply(
    lambda s: any(v in s for v in candidates)
)

gdf_dameritz = gdf_clean[mask].copy()
print("🔎 matches:", len(gdf_dameritz))
gdf_dameritz[["name","area_sqkm","centroid_lon","centroid_lat"]].head(10)


🔎 matches: 0


,name,area_sqkm,centroid_lon,centroid_lat


In [57]:
import pandas as pd
from pathlib import Path

# 1. Load the cleaned Dämeritzsee CSV
csv_path = Path("../sources/demeritzsee_clean.csv")
df = pd.read_csv(csv_path)

print("Loaded CSV from:", csv_path)
print("Columns in file:", df.columns.tolist())

# 2. Parse datetime if it exists
if "datetime" in df.columns:
    df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce")
else:
    print("⚠️ Warning: 'datetime' column not found, date stats will be skipped.")

# 3. Build stats dictionary safely
stats = {
    "n_rows": len(df),
}

# date range (only if datetime is available)
if "datetime" in df.columns:
    stats.update({
        "date_min": df["datetime"].min(),
        "date_max": df["datetime"].max(),
    })

# numeric columns we *expect* in the cleaned file
numeric_cols = [
    "temp_c",
    "conductivity_uS",
    "depth_m",
    "ph",
    "oxygen_saturation_pct",
    "oxygen_concentration_mgl",
    "secchi_depth_m",
]

for col in numeric_cols:
    if col in df.columns:
        stats[f"{col}_mean"] = round(df[col].mean(), 2)

stats


Loaded CSV from: ../sources/demeritzsee_clean.csv
Columns in file: ['13.625819492700556,52.400747119279224,OSM lakes', 'lake_name', 'source']
⚠️ Warning: 'datetime' column not found, date stats will be skipped.


{'n_rows': 271}

In [59]:
import pandas as pd
from pathlib import Path
from datetime import datetime

# Map to water_type
def infer_water_type(row):
    # OSM has 'natural' and 'water' tags; prefer 'water' value if present
    wt = (str(row.get("water", "")).strip().lower() or
          str(row.get("natural", "")).strip().lower())
    # Normalize a few common variants
    mapping = {
        "lake": "lake", "pond": "pond", "reservoir": "reservoir",
        "basin": "basin", "lagoon": "lagoon", "oxbow": "oxbow",
        "water": "lake"  # fallback
    }
    return mapping.get(wt, wt if wt else None)

gdf_u = gdf_clean.copy()

# area_ha from your area_sqkm
gdf_u["area_ha"] = (gdf_u["area_sqkm"] * 100).round(4)

# target columns
gdf_u["lake_name"] = gdf_u["name_norm"]
gdf_u["water_type"] = gdf_u.apply(infer_water_type, axis=1)
gdf_u["max_depth_m"] = pd.NA
gdf_u["perimeter_m"] = pd.NA
gdf_u["has_public_access"] = pd.NA
gdf_u["swimming_allowed"] = pd.NA
gdf_u["water_quality"] = pd.NA
gdf_u["data_source"] = "OSM waterbodies (Berlin)"
gdf_u["last_updated"] = pd.Timestamp(datetime.utcnow())

# order columns
cols = ["lake_name","geometry","centroid_lat","centroid_lon",
        "water_type","area_ha","max_depth_m","perimeter_m",
        "has_public_access","swimming_allowed","water_quality",
        "data_source","last_updated"]
gdf_u = gdf_u[cols]

# write outputs
out_geo = Path("../sources/lakes_berlin_unified.geojson")
out_csv = Path("../sources/berlin_lakes_summary.csv")  # keep name

gdf_u.to_file(out_geo, driver="GeoJSON")
gdf_u.drop(columns="geometry").to_csv(out_csv, index=False)

print("✅ Saved:", out_geo)
print("✅ Saved:", out_csv)


✅ Saved: ../sources/lakes_berlin_unified.geojson
✅ Saved: ../sources/berlin_lakes_summary.csv


/var/folders/kl/8b94v_052td7vx7hwzsc0fpw0000gn/T/ipykernel_12615/4099257948.py:32: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  gdf_u["last_updated"] = pd.Timestamp(datetime.utcnow())


In [60]:
# Cell A – rebuild lakes_berlin_unified.geojson safely

from pathlib import Path
import geopandas as gpd

SRC = Path("../sources").resolve()

clean_path = SRC / "berlin_lakes_clean.geojson"
u_path     = SRC / "lakes_berlin_unified.geojson"
summary_csv = SRC / "berlin_lakes_summary.csv"

# load the cleaned OSM lakes (we already know this file is OK)
gdf_clean = gpd.read_file(clean_path)
print("Loaded cleaned lakes:", gdf_clean.shape)

# pick the columns we actually want in the unified layer
cols_keep = [
    "name",
    "natural",
    "water",
    "wikidata",
    "area_sqkm",
    "centroid_lon",
    "centroid_lat",
    "geometry",
]

# some columns might not exist depending on earlier steps, so filter safely
cols_keep = [c for c in cols_keep if c in gdf_clean.columns]

gdf_u = gdf_clean[cols_keep].copy()

# standardize to 'lake_name'
if "name" in gdf_u.columns:
    gdf_u = gdf_u.rename(columns={"name": "lake_name"})

# add a simple source column
gdf_u["source"] = "OSM lakes"

print("Unified gdf_u shape:", gdf_u.shape)
print("Unified columns:", list(gdf_u.columns))

# save unified GeoJSON
gdf_u.to_file(u_path, driver="GeoJSON")
print("💾 Saved unified GeoJSON to:", u_path)

# save summary CSV (tabular metadata)
summary_cols = [
    "lake_name",
    "area_sqkm",
    "centroid_lon",
    "centroid_lat",
    "natural",
    "water",
    "wikidata",
    "source",
]
summary_cols = [c for c in summary_cols if c in gdf_u.columns]

gdf_u[summary_cols].to_csv(summary_csv, index=False)
print("💾 Saved unified summary CSV to:", summary_csv)


Loaded cleaned lakes: (241, 9)
Unified gdf_u shape: (241, 9)
Unified columns: ['lake_name', 'natural', 'water', 'wikidata', 'area_sqkm', 'centroid_lon', 'centroid_lat', 'geometry', 'source']
💾 Saved unified GeoJSON to: /Users/robertsesazi/Documents/layered-populate-data-pool-da/lakes/sources/lakes_berlin_unified.geojson
💾 Saved unified summary CSV to: /Users/robertsesazi/Documents/layered-populate-data-pool-da/lakes/sources/berlin_lakes_summary.csv


In [62]:
# Cell B – quality summary for unified layer

from pathlib import Path
import geopandas as gpd

SRC = Path("../sources").resolve()
u_path = SRC / "lakes_berlin_unified.geojson"

print("Loading unified lakes from:", u_path)
gdf_u = gpd.read_file(u_path)
print("Shape:", gdf_u.shape)
print("Columns:", list(gdf_u.columns))

# basic geometry quality
valid_pct = 100 * gdf_u.is_valid.mean()

summary = {
    "features": len(gdf_u),
    "valid_geometry_%": round(valid_pct, 1),
}

# only do name-based checks if we actually have lake_name
if "lake_name" in gdf_u.columns:
    missing_name_pct = 100 * gdf_u["lake_name"].isna().mean()
    dupe_names = gdf_u["lake_name"].str.lower().value_counts()
    dupe_count = int((dupe_names > 1).sum())

    summary.update({
        "missing_lake_name_%": round(missing_name_pct, 1),
        "duplicate_name_groups": dupe_count,
    })

summary


Loading unified lakes from: /Users/robertsesazi/Documents/layered-populate-data-pool-da/lakes/sources/lakes_berlin_unified.geojson
Shape: (241, 9)
Columns: ['lake_name', 'natural', 'water', 'wikidata', 'area_sqkm', 'centroid_lon', 'centroid_lat', 'source', 'geometry']


{'features': 241,
 'valid_geometry_%': np.float64(100.0),
 'missing_lake_name_%': np.float64(0.0),
 'duplicate_name_groups': 0}

In [67]:
from pathlib import Path
import json

SRC = Path("../sources").resolve()

raw_path = SRC / "osm_berlin_lakes_raw.json"
geojson_path = SRC / "osm_berlin_lakes.geojson"

print("Raw file exists:", raw_path.exists(), "size:", raw_path.stat().st_size, "bytes")
print("GeoJSON file exists:", geojson_path.exists(), "size:", geojson_path.stat().st_size, "bytes")

with raw_path.open() as f:
    raw_data = json.load(f)
print("Number of raw elements:", len(raw_data.get("elements", [])))

with geojson_path.open() as f:
    gj = json.load(f)
print("Number of GeoJSON features:", len(gj.get("features", [])))


Raw file exists: True size: 2030444 bytes
GeoJSON file exists: True size: 723421 bytes
Number of raw elements: 687
Number of GeoJSON features: 633


In [71]:
from pathlib import Path
import json

SRC = Path("../sources").resolve()
raw_path = SRC / "osm_berlin_lakes_raw.json"

with raw_path.open("r", encoding="utf-8") as f:
    raw_data = json.load(f)

# How many elements?
elements = raw_data.get("elements", [])
print("Number of raw elements:", len(elements))

# Show the first 2 elements (truncated) to see structure
for i, el in enumerate(elements[:2]):
    print(f"\n--- Element {i} ---")
    print("type:", el.get("type"))
    print("id:", el.get("id"))
    print("tags:", el.get("tags"))
    print("geometry length:", len(el.get("geometry", [])))


Number of raw elements: 687

--- Element 0 ---
type: way
id: 4317997
tags: {'intermittent': 'no', 'leisure': 'swimming_area', 'name': 'Schlachtensee', 'natural': 'water', 'postal_code': '14129', 'salt': 'no', 'sport': 'swimming', 'tidal': 'no', 'water': 'lake', 'wikidata': 'Q896095', 'wikipedia': 'de:Schlachtensee'}
geometry length: 216

--- Element 1 ---
type: way
id: 4401084
tags: {'name': 'Koenigssee', 'natural': 'water', 'water': 'lake', 'wikidata': 'Q1778289', 'wikipedia': 'de:Koenigssee'}
geometry length: 79


In [73]:
from pathlib import Path
import geopandas as gpd

SRC = Path("../sources").resolve()
geojson_path = SRC / "osm_berlin_lakes.geojson"

gdf_osm = gpd.read_file(geojson_path)

print("GeoDataFrame shape:", gdf_osm.shape)
print("Columns:", list(gdf_osm.columns))
print("\nFirst 5 rows:")
gdf_osm.head()


GeoDataFrame shape: (633, 5)
Columns: ['name', 'natural', 'water', 'wikidata', 'geometry']

First 5 rows:


,name,natural,water,wikidata,geometry
0,Schlachtensee,water,lake,Q896095,"POLYGON ((13.2281 52.44468, 13.22814 52.44463,..."
1,Koenigssee,water,lake,Q1778289,"POLYGON ((13.27181 52.48856, 13.27203 52.48864..."
2,Fennsee,water,lake,Q1404799,"POLYGON ((13.31548 52.48379, 13.31556 52.48379..."
3,Dreipfuhl,water,pond,None,"POLYGON ((13.27313 52.44797, 13.27316 52.44798..."
4,Lietzensee,water,lake,Q258569,"POLYGON ((13.289 52.50765, 13.28874 52.50756, ..."


In [74]:
print("RAW JSON:", raw_path, "size (bytes):", raw_path.stat().st_size)
print("GEOJSON:", geojson_path, "size (bytes):", geojson_path.stat().st_size)
print("Number of raw elements:", len(elements))
print("Number of GeoJSON features:", len(gdf_osm))


RAW JSON: /Users/robertsesazi/Documents/layered-populate-data-pool-da/lakes/sources/osm_berlin_lakes_raw.json size (bytes): 2030444
GEOJSON: /Users/robertsesazi/Documents/layered-populate-data-pool-da/lakes/sources/osm_berlin_lakes.geojson size (bytes): 723421
Number of raw elements: 687
Number of GeoJSON features: 633


#  Lakes Data Layer –  Data Transformation & Preprocessing  

This step covers the transformation and preprocessing of the **Berlin Lakes and Waterbodies** data layer.  
It integrates OpenStreetMap (OSM) water polygons with the Dämeritzsee in-situ dataset, harmonizes attributes, and prepares unified outputs in GeoJSON and CSV formats.

---

## 1 Input Sources  

| File | Description |
|------|--------------|
| `osm_berlin_lakes.geojson` | Raw OSM extract for all Berlin water polygons |
| `demeritzsee.csv` | In-situ measurements for the Dämeritzsee (temperature, pH, O₂, etc.) |

---

## 2 Transformation Pipeline  

All transformations are implemented in [`scripts/lakes_data_transformation.ipynb`](../scripts/lakes_data_transformation.ipynb).  
Below is a summary of the main processing steps:

1. **Load & inspect** raw OSM waterbody polygons.  
2. **Filter** relevant features (`natural=water`, `water=lake|pond|reservoir`).  
3. **Clean** geometry and attribute fields (removed empty names / duplicates).  
4. **Reproject** from `EPSG:4326` → `EPSG:25833` (for area m²) → back to `EPSG:4326`.  
5. **Compute metrics**  
   - `area_sqkm` → converted to `area_ha` (×100).  
   - `centroid_lon` / `centroid_lat`.  
6. **Harmonize columns** to the unified schema (see below).  
7. **Add metadata fields** (`data_source`, `last_updated`, placeholders for depth / access / swimming).  
8. **Export outputs**:  
   - `lakes_berlin_unified.geojson` → clean dataset with geometry.  
   - `berlin_lakes_summary.csv` → same table without geometry.  
9. **Compute QA metrics** (valid geometries, missing names, duplicates).  

---

## 3 Unified Dataset Schema Proposal  

| Column | Type | Description |
|:--|:--|:--|
| `lake_name` | VARCHAR | Official or normalized name of the lake or waterbody |
| `geometry` | GEOMETRY(POLYGON, 4326) | Polygon geometry in WGS 84 |
| `centroid_lat` | FLOAT | Latitude of lake centroid |
| `centroid_lon` | FLOAT | Longitude of lake centroid |
| `water_type` | VARCHAR | Classification (lake, pond, reservoir, basin etc.) |
| `area_ha` | FLOAT | Surface area in hectares |
| `max_depth_m` | FLOAT | Maximum depth (if available / null otherwise) |
| `perimeter_m` | FLOAT | Perimeter length (if available / null otherwise) |
| `has_public_access` | BOOLEAN | Public access flag (Yes/No/Unknown) |
| `swimming_allowed` | BOOLEAN | Swimming allowed flag (Yes/No/Unknown) |
| `water_quality` | VARCHAR | Water quality classification (if available) |
| `data_source` | VARCHAR | Data origin (e.g. OSM waterbodies, LUBW Atlas) |
| `last_updated` | TIMESTAMP | UTC timestamp of data export |

---

## 4 Output Files  

| File | Description |
|------|--------------|
| `lakes_berlin_unified.geojson` | Cleaned and standardized lake polygons (WGS 84 / EPSG:4326) |
| `berlin_lakes_summary.csv` | Tabular summary without geometry (for database loading preview) |
| *(Optional)* `demeritzsee_clean.csv` | Cleaned in-situ measurements for Dämeritzsee |
| *(Optional)* `daemeritzsee_polygon.geojson` | Extracted polygon for Dämeritzsee from OSM dataset |

---

## 5 Data Quality Summary  

| Metric | Value | Comment |
|:--|:--|:--|
| Total features | ≈ 276 | After filtering and deduplication |
| Valid geometry (%) | ≈ 99–100 | Validated using `GeoSeries.is_valid` |
| Missing lake_name (%) | < 5 | Mostly small unnamed ponds |
| Duplicate name groups | 0–2 | Minor naming variations |
| CRS | EPSG:4326 (WGS 84) | All geometry and centroids standardized |

*(Exact numbers printed in the notebook under `summary` cell.)*

---

## 6 Tools & Environment  

- **Python 3.13.5**  
- **pandas 2.2+**, **geopandas 0.14+**, **shapely**, **pyogrio / fiona**  
- Jupyter Notebook environment (VS Code)  

---

## 7 Reproducibility  

To reproduce all exports locally:
```bash
cd lakes/scripts
jupyter notebook lakes_data_transformation.ipynb



Lakes Data Layer – Data Transformation & Preprocessing 

This section documents how the Berlin Lakes & Waterbodies data layer is built.
It covers data ingestion from OpenStreetMap (OSM), cleaning, harmonization, metric calculations, and exporting unified datasets in GeoJSON and CSV formats.

1. Input Sources
File	Description
osm_berlin_lakes.geojson	Raw OSM extract containing all Berlin lake and pond polygons.
demeritzsee.csv	In-situ measurement dataset for the Dämeritzsee (temperature, pH, O₂, etc.).
2. Transformation Pipeline

The full transformation logic is implemented in
scripts/lakes_data_transformation.ipynb.

Processing steps:

Load OSM Waterbody Polygons

Reads osm_berlin_lakes.geojson produced from osm_berlin_lakes_raw.json.

Filter Relevant Features

Retains only natural=water and water ∈ {lake, pond}.

Clean Attributes

Drops unnamed waterbodies.

Removes duplicates.

Normalizes names (lowercase, accent-free).

Reproject Geometry

WGS84 → EPSG:25833 (for metric accuracy).

Compute area (area_sqkm → area_ha), centroids.

Reproject back to EPSG:4326.

Harmonize Columns

Standardized fields: lake_name, water_type, area_ha, centroid_lon, centroid_lat, etc.

Add Metadata Fields

data_source = "OSM waterbodies (Berlin)"

last_updated = <UTC timestamp>

Placeholders for depth, swimming access, water quality.

Export Outputs

GeoJSON: lakes_berlin_unified.geojson → includes geometry.

CSV: berlin_lakes_summary.csv → tabular view without geometry.

Quality Checks

Missing names, duplicate waterbodies, valid geometries.

3. Unified Dataset Schema
Column	Type	Description
lake_name	VARCHAR	Standardized lake or pond name.
geometry	POLYGON (EPSG:4326)	Lake boundary in WGS84.
centroid_lon	FLOAT	Longitudinal coordinate of centroid.
centroid_lat	FLOAT	Latitudinal coordinate of centroid.
water_type	VARCHAR	Lake / pond classification.
area_ha	FLOAT	Area in hectares.
max_depth_m	FLOAT (nullable)	Placeholder for in-situ depth data.
has_public_access	BOOLEAN (nullable)	Placeholder flag.
swimming_allowed	BOOLEAN (nullable)	Placeholder flag.
water_quality	VARCHAR (nullable)	Placeholder.
data_source	VARCHAR	Provenance of record.
last_updated	TIMESTAMP	Processing timestamp.